<a href="https://colab.research.google.com/github/sastha-06/24ADI006_DL_24BAD106/blob/main/24BAD106_DL_EX10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Name: Sastha Jeyasri A
# Roll Number: 24BAD106
# Part A: Environment setup

!pip uninstall -y torchvision
!pip install -U transformers gradio

Found existing installation: torchvision 0.26.0+cu130
Uninstalling torchvision-0.26.0+cu130:
  Successfully uninstalled torchvision-0.26.0+cu130
  Using cached transformers-5.19.0-py3-none-any.whl.metadata (32 kB)
  Using cached gradio-6.30.0-py3-none-any.whl.metadata (16 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 6.9 MB/s eta 0:00:00
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 2.7.1
    Uninstalling gradio_client-2.7.1:
      Successfully uninstalled gradio_client-2.7.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.18.0
    Uninstalling transformers-5.18.0:
      Successfully uninstalled transformers-5.18.0
  Attempting uninstall: gradio
    Found existing installation: gradio 6.29.0
    Uninstalling gradio-6.29.0:
      Successfully

In [3]:
# Name: Sastha Jeyasri A.
# Roll Number: 24BAD106
# Part A: Load and test the pretrained model

import torch
from transformers import pipeline

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
device = 0 if torch.cuda.is_available() else -1

print("Device:", "GPU" if device == 0 else "CPU")
print("Loading the Transformer model...")

chatbot_model = pipeline(
    "text-generation",
    model=MODEL_NAME,
    torch_dtype=torch.float16 if device == 0 else torch.float32,
    device=device
)

print("Model loaded successfully!")

setup_prompt = [{"role": "user", "content": "Explain what a chatbot is in one sentence."}]
setup_result = chatbot_model(setup_prompt, max_new_tokens=60, do_sample=False)
setup_generated = setup_result[0]["generated_text"]

if isinstance(setup_generated, list) and setup_generated:
    setup_answer = setup_generated[-1].get("content", "")
else:
    setup_answer = str(setup_generated)

print("\nSetup test prompt:", setup_prompt[-1]["content"])
print("Model response:", setup_answer)

Device: GPU
Loading the Transformer model...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

...

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Model loaded successfully!


[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



Setup test prompt: Explain what a chatbot is in one sentence.
Model response: A chatbot is a computer program that mimics human conversation and can interact with users in real-time, providing information, answering questions, and performing tasks.


## Part B: Prompt Handling and Response Generation
The function handles empty input, includes recent conversation history, generates a response, and handles runtime errors. It supports tuple-based history used by older Gradio versions.

In [4]:
#Name: Sastha Jeyasri A.
# Roll Number: 24BAD106
# Part B: Prompt handling and response generation

def generate_response(message, history=None):
    if not message or not message.strip():
        return "Please enter a valid message."

    messages = [{
        "role": "system",
        "content": "You are a helpful assistant. Give clear, concise answers."
    }]

    # Older Gradio format: (user_message, assistant_response) pairs
    for item in (history or [])[-6:]:
        if isinstance(item, (tuple, list)) and len(item) == 2:
            user_text, assistant_text = item
            if user_text:
                messages.append({"role": "user", "content": str(user_text)})
            if assistant_text:
                messages.append({"role": "assistant", "content": str(assistant_text)})
        elif isinstance(item, dict):
            role = item.get("role")
            content = item.get("content", "")
            if role in ("user", "assistant") and isinstance(content, str):
                messages.append({"role": role, "content": content})

    messages.append({"role": "user", "content": message.strip()})

    try:
        result = chatbot_model(
            messages,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )
        generated = result[0]["generated_text"]

        if isinstance(generated, list) and generated:
            answer = generated[-1].get("content", "")
        else:
            answer = str(generated)

        return answer.strip() or "No response generated."

    except Exception as error:
        print("Generation error:", error)
        return "Sorry, an error occurred while generating the response."


# Test at least five prompts
test_prompts = [
    "What is a Large Language Model?",
    "Explain the Transformer architecture simply.",
    "How is machine learning used in healthcare?",
    "Give three tips for effective studying.",
    "Why is environmental protection important?"
]

print("Student: Sastha Jeyasri A.")
print("Roll Number: 24BAD106")
print("\nPROMPT TESTING RESULTS")

for i, prompt in enumerate(test_prompts, start=1):
    print(f"\nTest {i}: {prompt}")
    print("Response:", generate_response(prompt))
    print("-" * 60)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Student: Sastha Jeyasri A.
Roll Number: 24BAD106

PROMPT TESTING RESULTS

Test 1: What is a Large Language Model?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Response: A Large Language Model (LLM) is a type of artificial intelligence (AI) model that can generate human-like language. It is designed to mimic the way humans process and generate language, and it can be used to create machine-translated text or to generate original text based on pre-existing text.

LLMs are often used in the fields of natural language processing (NLP), machine translation, and chatbots. They are particularly useful for creating large volumes of text, such as in translation, text generation, and dialogue systems.

LLMs are designed to be extremely efficient and can generate large volumes of text in a short amount of time. They are typically trained on large
------------------------------------------------------------

Test 2: Explain the Transformer architecture simply.


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Response: The Transformer architecture is a deep learning neural network architecture that uses a transformer to encode input data into a sequence of vectors called the encoder. The transformer then uses these vectors to decode the input data into a sequence of outputs called the decoder.

The transformer architecture consists of several key components:

1. Encoder: This component takes input tokens (words or characters) and generates a sequence of vectors (or latent representations) for those tokens. These vectors are called the encoder input vectors.

2. Decoder: This component takes the encoded input vectors from the encoder and generates a sequence of output tokens (words or characters). The decoder uses the same set of encoder output vectors as
------------------------------------------------------------

Test 3: How is machine learning used in healthcare?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Response: Machine learning is a branch of artificial intelligence (AI) that enables computers to learn from data without being programmed. In the healthcare industry, machine learning is used to analyze and interpret patient data, predict health outcomes, and improve healthcare delivery.

Here are some ways machine learning is used in healthcare:

1. Predictive Analytics: Machine learning algorithms are used to identify patterns and trends in medical data, such as hospital readmissions, medication adherence, and clinical outcomes. These insights can help healthcare providers make informed decisions, improve patient outcomes, and reduce costs.

2. Personalized Medicine: Machine learning algorithms can be used to analyze medical data
------------------------------------------------------------

Test 4: Give three tips for effective studying.


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Response: Sure, here are three tips for effective studying:

1. Plan your study time: Plan your study time in advance and stick to it. Set aside specific study hours for each subject, and avoid cramming. This will help you avoid overwhelming yourself and allow you to focus on each subject thoroughly.

2. Create a study environment: Create a study environment that is conducive to studying. This could be a quiet study space, a comfortable chair, or a designated study area. Make sure the environment is free from distractions, and the lighting is comfortable.

3. Use a study timer: Set a timer for each study session. This will help you stay focused and avoid distra
------------------------------------------------------------

Test 5: Why is environmental protection important?
Response: Environmental protection is important because it is crucial for the health and well-being of all living things on our planet. Protecting the environment is necessary to ensure the stability and resilience of

## Part C: Building the Chatbot Interface
This Gradio interface displays messages, retains conversation history, and includes a Clear Chat button. It omits `type="messages"` for compatibility with older Gradio versions.

In [6]:

#Name: Sastha Jeyasri A.
# Roll Number: 24BAD106
# Part C: Gradio chatbot interface with error debugging

import gradio as gr
import traceback

def respond(message, history):
    history = history or []

    if not message or not message.strip():
        return "", history

    user_message = message.strip()

    try:
        print("User prompt:", user_message)
        print("History received:", history)

        # Call the model function from Part B
        answer = generate_response(user_message, history)

        print("Generated response successfully.")

        updated_history = list(history)
        updated_history.append({
            "role": "user",
            "content": user_message
        })
        updated_history.append({
            "role": "assistant",
            "content": str(answer)
        })

        return "", updated_history

    except Exception as e:
        print("ERROR INSIDE generate_response():")
        traceback.print_exc()

        updated_history = list(history)
        updated_history.append({
            "role": "user",
            "content": user_message
        })
        updated_history.append({
            "role": "assistant",
            "content": f"Generation failed: {type(e).__name__}: {e}"
        })

        return "", updated_history


with gr.Blocks(title="Transformer-Based LLM Chatbot") as demo:
    gr.Markdown(
        "# Transformer-Based LLM Chatbot\n"
        "**Student:** Sastha Jeyasri A.  \n"
        "**Roll Number:** 24BAD106  \n"
        "Runs a locally loaded pretrained model without an API key."
    )

    chat = gr.Chatbot(label="Conversation", height=400)

    message = gr.Textbox(
        label="Enter your prompt",
        placeholder="Ask me anything...",
        lines=2
    )

    with gr.Row():
        send = gr.Button("Send", variant="primary")
        clear = gr.Button("Clear Chat")

    send.click(
        respond,
        inputs=[message, chat],
        outputs=[message, chat]
    )

    message.submit(
        respond,
        inputs=[message, chat],
        outputs=[message, chat]
    )

    clear.click(
        lambda: ("", []),
        inputs=[],
        outputs=[message, chat]
    )

demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c0c201b8b42232caf3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User prompt: what is machine learning?
History received: []
Generated response successfully.
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://87dd67f8092b38e4f2.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://c0c201b8b42232caf3.gradio.live


## Part D: Testing and Deployment
Test factual, conversational, and task-oriented prompts. Evaluate whether responses are relevant, readable, and reasonably accurate. Record actual observations after running the notebook; do not assume every response is correct.

In [7]:
#Name: Sastha Jeyasri A.
# Roll Number: 24BAD106
# Part D: Testing and evaluation

evaluation_prompts = [
    ("Factual", "What is self-attention in a Transformer?"),
    ("Conversational", "I am nervous about an upcoming exam. Give me two practical suggestions."),
    ("Task-oriented", "Write a short, polite email asking a teacher to clarify an assignment deadline."),
    ("Explanation", "Explain one benefit and one limitation of machine learning in healthcare."),
    ("Limitations", "Can an AI chatbot make mistakes? Explain briefly.")
]

print("CHATBOT EVALUATION")
print("Student: Sastha Jeyasri A.")
print("Roll Number: 24BAD106")

for i, (category, prompt) in enumerate(evaluation_prompts, start=1):
    answer = generate_response(prompt)
    print(f"\nTest case {i} | Category: {category}")
    print("Prompt:", prompt)
    print("Response:", answer)
    print("-" * 60)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


CHATBOT EVALUATION
Student: Sastha Jeyasri A.
Roll Number: 24BAD106


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Test case 1 | Category: Factual
Prompt: What is self-attention in a Transformer?
Response: Self-attention is a key feature of Transformer architecture, which is a type of neural network designed for natural language processing (NLP) tasks. In a Transformer, each layer of the network is composed of multiple self-attention blocks, which are designed to compute the attention weights for each input token in a sequence. These attention weights are used to weight the input tokens to produce a vector representation of the sequence. The attention mechanism allows the network to attend to specific parts of the input sequence and focus on the relevant information, improving the accuracy and efficiency of the NLP task.
------------------------------------------------------------


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Test case 2 | Category: Conversational
Prompt: I am nervous about an upcoming exam. Give me two practical suggestions.
Response: Certainly, here are two practical suggestions to help you feel more confident and prepared for an upcoming exam:

1. Start early: If possible, try to start studying for your exam as soon as possible. This will give your brain enough time to process and retain the information.

2. Practice: Take practice tests to familiarize yourself with the exam format and to identify areas where you may need to improve. This can help you to identify your strengths and weaknesses and to make adjustments to your study plan.

Remember, the key to success is preparation and practice. Don't be afraid to take breaks or to ask for help when you need it. Good luck on your exam!
------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Test case 3 | Category: Task-oriented
Prompt: Write a short, polite email asking a teacher to clarify an assignment deadline.
Response: Dear [Teacher's Name],

I hope this email finds you well. I am writing to request clarification on the deadline for your upcoming assignment. I understand that you have a lot on your plate, and I appreciate your efforts to ensure that your students receive quality work.

As you are aware, the deadline for this assignment is [insert date] by [insert time]. However, I would like to know if you have any specific instructions or requirements for the content or format of the assignment. I want to make sure that I am fully prepared to submit a high-quality product.

If you could please provide me with any additional details or guidelines, it would be greatly appreciated.
------------------------------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Test case 4 | Category: Explanation
Prompt: Explain one benefit and one limitation of machine learning in healthcare.
Response: Benefits of Machine Learning in Healthcare:
1. Predictive Analytics: Machine learning algorithms can be used to predict and prevent health problems before they occur. This can help in early detection and treatment of diseases, reducing the risk of complications and hospitalizations.

2. Personalized Care: Machine learning can be used to personalize medical treatments, medications, and surgical procedures. This can improve patient outcomes, reduce healthcare costs, and increase patient satisfaction.

Limitations of Machine Learning in Healthcare:
1. Data Scarcity: The availability of large amounts of healthcare data is often a limitation for machine learning algorithms. This can make it difficult to train and evaluate machine
------------------------------------------------------------

Test case 5 | Category: Limitations
Prompt: Can an AI chatbot make mistake